# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Lectura guiada · Machine Learning UTP
Cristian Alvarado · Ejecuta de arriba hacia abajo. No hay tareas ni entregas. Los ejemplos se revisan con el docente.

## Ruta de esta práctica · ML · Encuentro 3 · PCA

**En el aula:** cuatro centros ilustran cómo resumir dos medidas que cambian juntas. El explorador utiliza otros 22 puntos. **Aquí:** 200 registros simulados con tres medidas sin unidades físicas llamadas X1, X2 y X3. Es una ampliación del mismo procedimiento, no la misma tabla ni una evaluación de centros reales.

**Recorrido:** entender qué representa un punto → ver las tres medidas → ponerlas en una escala comparable → resumirlas en dos coordenadas → ver qué se perdería con una sola. Después, como detalle, reconstruir la coordenada de un registro.

En Colab guarda una copia personal en Drive. Ejecuta la celda principal y después la celda de comprobación, en ese orden. CPU es suficiente; no se requieren claves ni recursos de pago. Si la sesión se reinicia, vuelve a ejecutar desde el comienzo.

### PCA y sus coordenadas, sin confundir porcentajes

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **PCA** — PCA: Principal Component Analysis (Análisis de componentes principales). Método que combina medidas numéricas en nuevas direcciones perpendiculares entre sí, ordenadas para conservar tanta variación como sea posible. Trabaja con datos centrados; decidir si se estandarizan antes es otra operación. El porcentaje conservado describe variación de las entradas, no aciertos de clasificación ni utilidad garantizada para predecir. [Fuente](https://scikit-learn.org/stable/modules/decomposition.html#pca)

- **PC1 y PC2** — PC1 y PC2: Principal Component 1 / Principal Component 2 (Primera y segunda componente principal). PC1 es la primera nueva dirección de PCA y PC2 la siguiente, perpendicular a la primera. La coordenada de un caso se obtiene al combinar sus medidas preparadas con los coeficientes de esa dirección. Son ejes calculados, no columnas físicas originales ni categorías. Los coeficientes pueden cambiar de signo junto con las coordenadas sin cambiar la representación geométrica; al hablar de «cargas» hay que declarar la convención. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

- **Varianza** — Varianza: Dispersión cuadrática respecto a la media. Resume cuánto se separan los valores de su media: se elevan al cuadrado las diferencias y se suman. La varianza poblacional divide por n; la estimación muestral habitual divide por n−1, con n mayor que uno. Tiene las unidades originales al cuadrado y vale cero si todos los valores son iguales. Debe declararse el denominador: StandardScaler usa n y explained_variance_ de PCA usa n−1; las proporciones coinciden si el mismo denominador se cancela. [Fuente](https://numpy.org/doc/stable/reference/generated/numpy.var.html)

- **Varianza explicada** — Varianza explicada: Variación conservada por las componentes. En PCA, una componente conserva una cantidad de varianza. Su proporción explicada divide esa cantidad entre la varianza total de las variables preparadas; varias componentes conservan la suma de sus proporciones. 0,90 significa 90 % de esa varianza; no significa 90 % de aciertos ni garantiza conservar la información útil para predecir. Hay que declarar el centrado y escalado aplicados. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)


## PCA: guardar menos números para representar los mismos casos

**Situación para entender la pregunta:** imagina muchas fichas y varias medidas en cada ficha. Si dos medidas cuentan casi lo mismo, quizás puedas dibujar las diferencias entre fichas con menos números. A ese resumen se dedica PCA. No está agrupando ni adivinando una respuesta correcta.

En esta simulación, cada ficha es un registro inventado. X1 y X2 se construyen para cambiar juntas, con algo de variación; X3 se genera por separado. No llamamos a X1 lluvia ni a X2 calidad: el programa no contiene esas mediciones. La semejanza con las fichas del aula está en el método, no en un significado agrícola de estos valores.

**Tres nombres antes del código:** variable = una medida por ficha; componente = nueva medida calculada mezclando las originales; dimensión = cuántos números necesitas para ubicar una ficha. De tres dimensiones a dos significa pasar de tres coordenadas por registro a dos. El registro sigue siendo el mismo.

### Antes de mirar fórmulas: qué cambia al resumir

Primero dibujamos un punto por registro en tres ejes. Después los mismos 200 puntos se dibujan con dos números, PC1 y PC2, y finalmente con uno, PC1. **PC** significa componente principal. No son medidas recogidas de nuevo: se calculan con X1, X2 y X3.

**Estandarizar** pone cada variable en relación con su propio promedio y su dispersión, para que sus tamaños numéricos no dominen el resumen. La cuenta de una variable es (valor−media)/desviación. Con valor 60, media 50 y desviación 10 daría 1: una desviación por encima de la media. Esos tres números son un ejemplo de lectura, no estadísticas calculadas de esta simulación.

**Varianza** resume cuánto difieren los números respecto a su promedio. La **varianza conservada** informa cuánto de esa dispersión queda en el resumen. Si dos componentes conservaran 90 %, se descartaría 10 % de la variación de las variables preparadas. Eso no es 90 % de predicciones correctas. Lee el porcentaje que calcula tu ejecución, no uses el 100 % del ejemplo diagonal del aula.

**Para leer la tabla de pesos:** cada fila corresponde a una componente y cada columna a una variable original ya estandarizada. Un peso es el multiplicador que le da participación. Las direcciones son ortogonales: están a 90° entre sí en el espacio preparado. La multiplicación y suma se abrirá después con un solo registro, no necesitas interpretar toda la matriz a la vez.

In [ ]:
# ================================================================
# REDUCCIÓN DE DIMENSIONALIDAD CON PCA (Colab listo para pegar)
# Caso simple: dataset sintético de 3 variables
# ================================================================
# ¿Qué hace este notebook?
# 1) Genera un dataset con 3 variables correlacionadas.
# 2) Visualiza los datos en 3D.
# 3) Aplica PCA para reducir a 2D y a 1D.
# 4) Explica cómo leer los resultados: varianza explicada, componentes, proyecciones.
#
# ------------------------------------------------
# ¿Qué es PCA?
# ------------------------------------------------
# PCA = Principal Component Analysis.
# Objetivo: encontrar nuevas variables (componentes principales) que:
# - Son combinaciones lineales de las originales.
# - La primera maximiza varianza; las siguientes son ortogonales y capturan la restante.
# - Para un número fijo de dimensiones, minimizan el error de reconstrucción cuadrático.
# - La varianza conservada no garantiza conservar lo relevante para otra tarea.
#
# Ejemplo:
# - Si lluvia y humedad están muy correlacionadas, PCA crea una sola
#   componente que resume ambas, reduciendo redundancia.
# ================================================================

from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D

# ----------------------------
# 1) Dataset sintético
# ----------------------------
# Creamos 3 variables con correlaciones:
# - X1 ~ normal
# - X2 ~ X1 con ruido (correlación alta)
# - X3 ~ independiente
rng = np.random.default_rng(42)
n = 200
X1 = rng.normal(50, 10, n)
X2 = X1 * 0.8 + rng.normal(0, 5, n) + 10
X3 = rng.normal(30, 8, n)

df = pd.DataFrame({"X1": X1, "X2": X2, "X3": X3})
display(df.head())

# ----------------------------
# 2) Visualización en 3D
# ----------------------------
fig = plt.figure(figsize=(6,5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df["X1"], df["X2"], df["X3"], alpha=0.6)
ax.set_xlabel("X1")
ax.set_ylabel("X2")
ax.set_zlabel("X3")
ax.set_title("Datos originales en 3D")
plt.show()

# ----------------------------
# 3) Escalado
# ----------------------------
# PCA es sensible a escala → StandardScaler.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# ----------------------------
# 4) PCA
# ----------------------------
pca = PCA(n_components=3)  # tantas como originales para analizar
X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada por cada componente:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"Componente {i+1}: {v:.2%}")

print("\nVarianza acumulada:", np.cumsum(pca.explained_variance_ratio_))

print("\nComponentes (pesos aplicados a cada variable estandarizada en cada PC):")
print(pd.DataFrame(pca.components_, columns=df.columns, index=[f"PC{i+1}" for i in range(3)]))

# ----------------------------
# 5) Visualización PCA 2D
# ----------------------------
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], alpha=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Datos reducidos a 2D con PCA")
plt.grid(True)
plt.show()

# ----------------------------
# 6) Visualización PCA 1D
# ----------------------------
plt.figure(figsize=(6,2))
plt.scatter(X_pca[:,0], np.zeros_like(X_pca[:,0]), alpha=0.6)
plt.xlabel("PC1 (única dimensión)")
plt.yticks([])
plt.title("Datos reducidos a 1D con PCA")
plt.show()

# ----------------------------
# 7) Interpretación
# ----------------------------
print(f"""
================= CÓMO LEER LOS RESULTADOS =================
1) Varianza explicada:
   - PC1 captura la mayor proporción de varianza; consulta el porcentaje calculado.
   - PC2 añade otra parte significativa.
   - PC3 suele explicar poco si hay redundancia entre variables.
   → Decisión: consulta la varianza acumulada calculada para decidir cuántas componentes conservar.

2) Componentes (matriz de pesos):
   - Cada PC combina z1, z2, z3: zj = (Xj - mediaj) / desviaciónj.
   - PC1 = w1*z1 + w2*z2 + w3*z3; los pesos de la tabla actúan tras escalar.
   - Si PC1 tiene coeficientes altos y parecidos en X1 y X2,
     significa que resume la "información común" entre ambas.

3) Gráfica 2D:
   - PC1 y PC2 conservan {pca.explained_variance_ratio_[:2].sum():.2%} de la varianza estandarizada.
   - Se descarta {1-pca.explained_variance_ratio_[:2].sum():.2%}; no son las distancias originales sin escalar.

4) Gráfica 1D:
   - PC1 conserva {pca.explained_variance_ratio_[0]:.2%} y descarta {1-pca.explained_variance_ratio_[0]:.2%} de varianza.
   - Una sola dimensión puede perder estructura relevante: depende de la tarea.

5) Uso práctico:
   - PCA reduce variables redundantes.
   - Puede ayudar a visualizar y preparar datos para otros algoritmos.
   - Una mejora de clustering o regresión requiere comparación; no está garantizada.
=============================================================
""")


### Lee las figuras en este orden

**Primera figura, tres ejes:** cada punto tiene X1, X2 y X3. Es la lista original de 200 registros. Rotar mentalmente el dibujo no modifica los datos.

**Segunda figura, PC1 y PC2:** cada punto sigue siendo uno de esos registros, pero sus coordenadas son dos combinaciones de las variables preparadas. Busca en la salida «PC1 y PC2 conservan…»: ese porcentaje se calcula sumando las proporciones de ambas componentes. Su complemento hasta 100 % es lo descartado.

**Tercera figura, solo PC1:** los puntos se colocan sobre una línea. El eje vertical vale cero únicamente para dibujarlos; no significa que otra medida real sea cero. Dos puntos pueden caer cerca porque la información que los distinguía se descartó.

**Prueba pequeña sin cambiar datos:** compara en la salida el porcentaje de PC1 con el de PC1+PC2. El segundo no puede ser menor porque añade una componente. Una componente necesita un número por registro y dos necesitan dos: estás comparando tamaño del resumen frente a información conservada.

### Una ficha, una coordenada: seguir los números sin leer toda la matriz

`df.iloc[[0]]` toma únicamente el primer registro para poder seguirlo; no vuelve a ajustar PCA. La celda aplica tres pasos con los valores ya aprendidos:

1. `scaler.transform` resta a cada medida su media y divide por su desviación. La tabla muestra valor original, media y desviación para que puedas comprobarlo.
2. `pca.mean_` es el centro aprendido por PCA. Se resta al registro estandarizado; aquí es muy cercano a cero porque ya se estandarizó.
3. Cada número centrado se multiplica por el peso de PC1 situado en la misma columna. Los tres productos se suman: el resultado es la posición de este registro en el eje PC1.

**Cómo leer la salida:** recorre una variable de izquierda a derecha: Original → Media y Desviación → valor preparado → Peso PC1 → Producto. Después mira el texto bajo la tabla: «suma de productos» y «coordenada del gráfico» deben coincidir, salvo el redondeo de pantalla. `assert` es una comprobación automática: no imprime nada si coincide y detiene la celda si no.

**Tip:** un peso negativo no significa una medida mala; indica el sentido de su aporte al eje. El signo completo de una componente puede invertirse sin cambiar la varianza conservada. No redondees los números usados para la cuenta: redondea solo lo que muestras.

In [ ]:
fila_original = df.iloc[[0]]
fila_z = scaler.transform(fila_original)[0]
fila_centrada = fila_z - pca.mean_
productos = fila_centrada * pca.components_[0]
display(pd.DataFrame({'Variable':df.columns,'Original':fila_original.iloc[0].values,
                      'Media':scaler.mean_,'Desviación':scaler.scale_,
                      'z centrado':fila_centrada,'Peso PC1':pca.components_[0],
                      'Producto':productos}))
print('PC1 = suma de productos =',productos.sum(),'· coordenada del gráfico =',X_pca[0,0])
assert np.isclose(productos.sum(),X_pca[0,0])
